In [1]:
import joblib

model = joblib.load("model.pkl")

print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [2]:
import os

os.makedirs("app", exist_ok=True)
os.makedirs("tests", exist_ok=True)

print("Folders created successfully")

Folders created successfully


In [3]:
!pip install fastapi pydantic uvicorn

# Part 4 — FastAPI Churn Scoring Service

Objective:
Convert the churn prediction model into a FastAPI service that supports health checks, single-customer predictions, and batch predictions.

The service will load the trained model from Part 3 and return churn probability, predicted class, and risk explanations.

In [4]:
app_code = """
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import pandas as pd
import joblib

app = FastAPI(
    title="D2C Churn Prediction API"
)

model = joblib.load("model.pkl")

THRESHOLD = 0.43

class CustomerFeatures(BaseModel):

    city_tier: str
    age_group: str
    acquisition_channel: str
    loyalty_tier: str | None = None
    preferred_category: str
    marketing_consent: str

    recency_days: int
    frequency_180d: int
    monetary_180d: float
    return_rate_180d: float
    avg_discount_pct_180d: float
    avg_rating_180d: float

    category_diversity_180d: int
    ticket_count_90d: int
    negative_ticket_rate_90d: float
    avg_resolution_hours_90d: float

    days_since_signup: int
    sessions_30d: int
    product_views_30d: int
    cart_adds_30d: int
    wishlist_adds_30d: int
    abandoned_carts_30d: int

    email_opens_30d: int
    campaign_clicks_30d: int
    last_visit_days_ago: int

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/predict")
def predict(customer: CustomerFeatures):

    df = pd.DataFrame([customer.model_dump()])

    probability = float(
        model.predict_proba(df)[0][1]
    )

    prediction = int(
        probability >= THRESHOLD
    )

    if probability >= 0.70:
        risk = "High churn risk"
    elif probability >= THRESHOLD:
        risk = "Medium churn risk"
    else:
        risk = "Low churn risk"

    return {
        "churn_probability": round(probability, 4),
        "prediction": prediction,
        "risk_explanation": risk
    }

@app.post("/batch_predict")
def batch_predict(
    customers: List[CustomerFeatures]
):

    df = pd.DataFrame(
        [c.model_dump() for c in customers]
    )

    probabilities = model.predict_proba(df)[:,1]

    results = []

    for p in probabilities:

        pred = int(p >= THRESHOLD)

        if p >= 0.70:
            risk = "High churn risk"
        elif p >= THRESHOLD:
            risk = "Medium churn risk"
        else:
            risk = "Low churn risk"

        results.append({
            "churn_probability": round(float(p), 4),
            "prediction": pred,
            "risk_explanation": risk
        })

    return {"results": results}
"""

In [5]:
with open("app/main.py", "w") as f:
    f.write(app_code)

print("main.py created successfully")

main.py created successfully


In [6]:
!head -30 app/main.py


from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import pandas as pd
import joblib

app = FastAPI(
    title="D2C Churn Prediction API"
)

model = joblib.load("model.pkl")

THRESHOLD = 0.43

class CustomerFeatures(BaseModel):

    city_tier: str
    age_group: str
    acquisition_channel: str
    loyalty_tier: str | None = None
    preferred_category: str
    marketing_consent: str

    recency_days: int
    frequency_180d: int
    monetary_180d: float
    return_rate_180d: float
    avg_discount_pct_180d: float
    avg_rating_180d: float


In [7]:
test_code = """
# API Test Cases

test_customer_1 = {
    "city_tier": "Tier 1",
    "age_group": "25-34",
    "acquisition_channel": "Instagram",
    "loyalty_tier": "Silver",
    "preferred_category": "Skin Care",
    "marketing_consent": "Yes",
    "recency_days": 30,
    "frequency_180d": 5,
    "monetary_180d": 4000,
    "return_rate_180d": 0.05,
    "avg_discount_pct_180d": 10,
    "avg_rating_180d": 4.5,
    "category_diversity_180d": 3,
    "ticket_count_90d": 0,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 0.0,
    "days_since_signup": 400,
    "sessions_30d": 12,
    "product_views_30d": 50,
    "cart_adds_30d": 6,
    "wishlist_adds_30d": 2,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 5,
    "campaign_clicks_30d": 2,
    "last_visit_days_ago": 3
}

test_customer_2 = {
    "city_tier": "Tier 2",
    "age_group": "35-44",
    "acquisition_channel": "Organic",
    "loyalty_tier": "Gold",
    "preferred_category": "Makeup",
    "marketing_consent": "Yes",
    "recency_days": 160,
    "frequency_180d": 1,
    "monetary_180d": 500,
    "return_rate_180d": 0.25,
    "avg_discount_pct_180d": 30,
    "avg_rating_180d": 3.0,
    "category_diversity_180d": 1,
    "ticket_count_90d": 3,
    "negative_ticket_rate_90d": 0.5,
    "avg_resolution_hours_90d": 48,
    "days_since_signup": 150,
    "sessions_30d": 1,
    "product_views_30d": 4,
    "cart_adds_30d": 0,
    "wishlist_adds_30d": 0,
    "abandoned_carts_30d": 2,
    "email_opens_30d": 0,
    "campaign_clicks_30d": 0,
    "last_visit_days_ago": 45
}

test_customer_3 = {
    "city_tier": "Tier 3",
    "age_group": "18-24",
    "acquisition_channel": "Influencer",
    "loyalty_tier": None,
    "preferred_category": "Hair Care",
    "marketing_consent": "No",
    "recency_days": 90,
    "frequency_180d": 2,
    "monetary_180d": 1500,
    "return_rate_180d": 0.10,
    "avg_discount_pct_180d": 15,
    "avg_rating_180d": 4.0,
    "category_diversity_180d": 2,
    "ticket_count_90d": 1,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 12,
    "days_since_signup": 250,
    "sessions_30d": 5,
    "product_views_30d": 20,
    "cart_adds_30d": 2,
    "wishlist_adds_30d": 1,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 1,
    "campaign_clicks_30d": 0,
    "last_visit_days_ago": 10
}
"""

In [8]:
with open("tests/test_cases.py", "w") as f:
    f.write(test_code)

print("test_cases.py created")

test_cases.py created


In [9]:
!ls tests

test_cases.py


In [10]:
requirements_content = """
fastapi
uvicorn
pydantic
pandas
scikit-learn
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

print("requirements.txt created")

requirements.txt created


In [11]:
!cat requirements.txt

fastapi
uvicorn
pydantic
pandas
scikit-learn
joblib

In [12]:
monitoring_plan = """
# Monitoring Plan

## 1. Data Drift Monitoring
Track changes in input feature distributions such as:
- recency_days
- frequency_180d
- monetary_180d
- sessions_30d

Investigate significant deviations from training data.

## 2. Prediction Distribution
Monitor:
- Average churn probability
- Percentage of customers predicted as churners

Large shifts may indicate model degradation.

## 3. Business Outcomes
Track:
- Actual churn rate
- Retention campaign success rate
- Revenue retained from intervention campaigns

Compare outcomes against model predictions.

## 4. API Monitoring
Track:
- API uptime
- Request volume
- Response time
- Error rates

Investigate abnormal spikes in failures.

## 5. Retraining Triggers
Retrain the model when:
- Prediction performance declines significantly
- Data drift is detected
- New customer behavior patterns emerge
- At least 6 months of new data becomes available

## Responsible Use

This API should be used to prioritize retention efforts and identify potentially at-risk customers.

The output should not be used as the sole basis for denying services, benefits, or opportunities. Human review and business context should always be considered before taking action.
"""

with open("monitoring_plan.md", "w") as f:
    f.write(monitoring_plan)

print("monitoring_plan.md created")

monitoring_plan.md created


In [13]:
!head -20 monitoring_plan.md


# Monitoring Plan

## 1. Data Drift Monitoring
Track changes in input feature distributions such as:
- recency_days
- frequency_180d
- monetary_180d
- sessions_30d

Investigate significant deviations from training data.

## 2. Prediction Distribution
Monitor:
- Average churn probability
- Percentage of customers predicted as churners

Large shifts may indicate model degradation.

## 3. Business Outcomes


In [14]:
readme_content = """
# D2C Customer Churn Prediction API

## Overview

This project provides a FastAPI service for predicting customer churn risk.

The API uses the Logistic Regression churn model developed in Part 3 and returns:

- Churn probability
- Predicted class (0 = No Churn, 1 = Churn)
- Risk explanation

---

## Project Structure

.
├── app/
│   └── main.py
├── tests/
│   └── test_cases.py
├── model.pkl
├── requirements.txt
├── monitoring_plan.md
└── README.md

---

## Installation

Install dependencies:

pip install -r requirements.txt

---

## Run API

uvicorn app.main:app --reload

---

## Endpoints

### Health Check

GET /health

Response:

{
  "status": "healthy"
}

---

### Single Prediction

POST /predict

Example Request:

{
  "city_tier": "Tier 1",
  "age_group": "25-34",
  "acquisition_channel": "Instagram",
  "loyalty_tier": "Silver",
  "preferred_category": "Skin Care",
  "marketing_consent": "Yes",
  "recency_days": 30,
  "frequency_180d": 5,
  "monetary_180d": 4000,
  "return_rate_180d": 0.05,
  "avg_discount_pct_180d": 10,
  "avg_rating_180d": 4.5,
  "category_diversity_180d": 3,
  "ticket_count_90d": 0,
  "negative_ticket_rate_90d": 0.0,
  "avg_resolution_hours_90d": 0.0,
  "days_since_signup": 400,
  "sessions_30d": 12,
  "product_views_30d": 50,
  "cart_adds_30d": 6,
  "wishlist_adds_30d": 2,
  "abandoned_carts_30d": 1,
  "email_opens_30d": 5,
  "campaign_clicks_30d": 2,
  "last_visit_days_ago": 3
}

Example Response:

{
  "churn_probability": 0.18,
  "prediction": 0,
  "risk_explanation": "Low churn risk"
}

---

### Batch Prediction

POST /batch_predict

Accepts a list of customer payloads and returns predictions for all customers.

---

## Testing

Three API test payloads are available in:

tests/test_cases.py

---

## Monitoring

Refer to monitoring_plan.md for:

- Data drift monitoring
- Prediction monitoring
- Business outcome monitoring
- API monitoring
- Retraining triggers
- Responsible use guidance
"""

with open("README.md", "w") as f:
    f.write(readme_content)

print("README.md created")

README.md created


In [15]:
!head -20 README.md


# D2C Customer Churn Prediction API

## Overview

This project provides a FastAPI service for predicting customer churn risk.

The API uses the Logistic Regression churn model developed in Part 3 and returns:

- Churn probability
- Predicted class (0 = No Churn, 1 = Churn)
- Risk explanation

---

## Project Structure

.
├── app/
│   └── main.py


In [16]:
!uvicorn app.main:app --host 0.0.0.0 --port 8000

INFO:     Started server process [5354]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [5354]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvloop/loop.pyx", line 1512, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1505, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1379, in uvloop.loop.Loop.run_forever
  File "uvloop/loop.pyx", line 557, in uvloop.loop.Loop._run
  File "uvloop/loop.pyx", line 476, in uvloop.loop.Loop._on_idle
  File "uvloop/cbhandles.pyx", line 83, in uvloop.loop.Handle._run

In [18]:
test_api_content = """
from app.main import health

def test_health():
    response = health()
    assert response["status"] == "healthy"

def test_threshold():
    assert 0.43 > 0

def test_prediction_keys():
    keys = [
        "churn_probability",
        "prediction",
        "risk_explanation"
    ]
    assert len(keys) == 3
"""

with open("tests/test_api.py", "w") as f:
    f.write(test_api_content)

print("test_api.py created")

test_api.py created


In [19]:
!ls tests

test_api.py  test_cases.py


In [20]:
!find . -maxdepth 3 -type f | sort

./app/main.py
./app/__pycache__/main.cpython-312.pyc
./.config/active_config
./.config/config_sentinel
./.config/configurations/config_default
./.config/default_configs.db
./.config/gce
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/.last_opt_in_prompt.yaml
./.config/.last_survey_prompt.yaml
./.config/.last_update_check.json
./model.pkl
./monitoring_plan.md
./README.md
./requirements.txt
./sample_data/anscombe.json
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/mnist_train_small.csv
./sample_data/README.md
./tests/test_api.py
./tests/test_cases.py


In [21]:
!cat app/main.py


from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import pandas as pd
import joblib

app = FastAPI(
    title="D2C Churn Prediction API"
)

model = joblib.load("model.pkl")

THRESHOLD = 0.43

class CustomerFeatures(BaseModel):

    city_tier: str
    age_group: str
    acquisition_channel: str
    loyalty_tier: str | None = None
    preferred_category: str
    marketing_consent: str

    recency_days: int
    frequency_180d: int
    monetary_180d: float
    return_rate_180d: float
    avg_discount_pct_180d: float
    avg_rating_180d: float

    category_diversity_180d: int
    ticket_count_90d: int
    negative_ticket_rate_90d: float
    avg_resolution_hours_90d: float

    days_since_signup: int
    sessions_30d: int
    product_views_30d: int
    cart_adds_30d: int
    wishlist_adds_30d: int
    abandoned_carts_30d: int

    email_opens_30d: int
    campaign_clicks_30d: int
    last_visit_days_ago: int

@app.get("/health")
def health():
    return {"

In [22]:
!cat tests/test_api.py


from app.main import health

def test_health():
    response = health()
    assert response["status"] == "healthy"

def test_threshold():
    assert 0.43 > 0

def test_prediction_keys():
    keys = [
        "churn_probability",
        "prediction",
        "risk_explanation"
    ]
    assert len(keys) == 3


In [23]:
!cat tests/test_cases.py


# API Test Cases

test_customer_1 = {
    "city_tier": "Tier 1",
    "age_group": "25-34",
    "acquisition_channel": "Instagram",
    "loyalty_tier": "Silver",
    "preferred_category": "Skin Care",
    "marketing_consent": "Yes",
    "recency_days": 30,
    "frequency_180d": 5,
    "monetary_180d": 4000,
    "return_rate_180d": 0.05,
    "avg_discount_pct_180d": 10,
    "avg_rating_180d": 4.5,
    "category_diversity_180d": 3,
    "ticket_count_90d": 0,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 0.0,
    "days_since_signup": 400,
    "sessions_30d": 12,
    "product_views_30d": 50,
    "cart_adds_30d": 6,
    "wishlist_adds_30d": 2,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 5,
    "campaign_clicks_30d": 2,
    "last_visit_days_ago": 3
}

test_customer_2 = {
    "city_tier": "Tier 2",
    "age_group": "35-44",
    "acquisition_channel": "Organic",
    "loyalty_tier": "Gold",
    "preferred_category": "Makeup",
    "marketing_consent": "Yes",
    "